In [1]:
import numpy as np
import pandas as pd
import joblib
import os
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED, N_MELS

Elegimos dataset

In [2]:
dataset = 'crudos'

# Datos originales

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [5]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((5181, 47616), (5181,), (1296, 47616), (1296,))

### Random Forest

#### Entrenamiento

In [6]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': randint(5, 15),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(10, 20)
}

In [7]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.741 total time=   7.2s
[CV 2/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.743 total time=   9.6s
[CV 3/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.742 total time=   9.6s
[CV 4/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.758 total time=  10.4s
[CV 5/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.756 total time=  10.5s
[CV 1/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.739 total time=   9.9s
[CV 2/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.742 total time=  14.1s
[CV 3/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.736 total time=  16.0s
[CV 4/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.755 total time=  16.1s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0020D877BE900>, 'min_samples_leaf': <scipy.stats....0020D87810550>, 'min_samples_split': <scipy.stats....0020D878107D0>}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [8]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
10,14,12,26,0.753246
15,13,11,24,0.751234
2,14,12,21,0.750439
9,13,10,25,0.749669
11,11,13,23,0.748358
0,11,13,27,0.748220
8,14,15,27,0.747898
6,12,15,16,0.747684
14,11,11,18,0.747329
16,13,19,19,0.746488


In [9]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 14, 'min_samples_leaf': 12, 'min_samples_split': 26}
Best CV score: 0.7532458756505369


In [10]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.93      0.92      2652
           1       0.93      0.91      0.92      2529

    accuracy                           0.92      5181
   macro avg       0.92      0.92      0.92      5181
weighted avg       0.92      0.92      0.92      5181



#### Evaluación

In [11]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.68      0.69       663
           1       0.68      0.69      0.68       633

    accuracy                           0.69      1296
   macro avg       0.69      0.69      0.69      1296
weighted avg       0.69      0.69      0.69      1296



#### Guardado

In [12]:
os.makedirs(f'./modelos_clasicos/modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/crudos/melspec_rf.pkl']

### Gradient Boost

In [11]:
gbc = GradientBoostingClassifier(
    n_estimators=500,
    random_state=SEED,
    n_iter_no_change=10,
    max_features='sqrt'
    )

In [12]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2]
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END ................learning_rate=0.05;, score=0.726 total time= 1.2min
[CV 2/5] END ................learning_rate=0.05;, score=0.755 total time= 2.8min
[CV 3/5] END ................learning_rate=0.05;, score=0.746 total time= 1.7min
[CV 4/5] END ................learning_rate=0.05;, score=0.739 total time=  58.7s
[CV 5/5] END ................learning_rate=0.05;, score=0.770 total time= 2.3min
[CV 1/5] END .................learning_rate=0.1;, score=0.726 total time= 1.2min
[CV 2/5] END .................learning_rate=0.1;, score=0.740 total time= 1.1min
[CV 3/5] END .................learning_rate=0.1;, score=0.731 total time=  51.8s
[CV 4/5] END .................learning_rate=0.1;, score=0.757 total time= 1.1min
[CV 5/5] END .................learning_rate=0.1;, score=0.770 total time= 1.6min
[CV 1/5] END .................learning_rate=0.2;, score=0.749 total time= 1.2min
[CV 2/5] END .................learning_rate=0.2;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.05, 0.1, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [13]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_learning_rate,mean_test_score
0,0.05,0.747095
1,0.10,0.744499
2,0.20,0.740088


In [14]:
best_model = rnd_search.best_estimator_

In [15]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.83      0.83      0.83      2652
           1       0.82      0.82      0.82      2529

    accuracy                           0.83      5181
   macro avg       0.83      0.83      0.83      5181
weighted avg       0.83      0.83      0.83      5181



In [16]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.68      0.68      0.68       663
           1       0.66      0.66      0.66       633

    accuracy                           0.67      1296
   macro avg       0.67      0.67      0.67      1296
weighted avg       0.67      0.67      0.67      1296



In [17]:
joblib.dump(gbc, f'./modelos_clasicos/modelos/{dataset}/melspec_gbc.pkl')

['./modelos_clasicos/modelos/crudos/melspec_gbc.pkl']

## Features de Audio

In [13]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [14]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((5181, 46), (5181,), (1296, 46), (1296,))

### Random Forest

#### Entrenamiento

In [15]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(10, 20),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(5, 15),
    'max_features': ['sqrt', 'log2', None]
}

In [16]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=16, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.834 total time=   0.4s
[CV 2/5] END max_depth=16, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.811 total time=   0.5s
[CV 3/5] END max_depth=16, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.849 total time=   0.4s
[CV 4/5] END max_depth=16, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.849 total time=   0.5s
[CV 5/5] END max_depth=16, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.830 total time=   0.5s
[CV 1/5] END max_depth=14, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.850 total time=   3.4s
[CV 2/5] END max_depth=14, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.819 total time=   4.7s
[CV 3/5] END max_depth=14, max_features=None, min_samples_leaf=14, min_samples_split=17;, s

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0020D8772FE10>, 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....0020D877A1FD0>, 'min_samples_split': <scipy.stats....0020D8772E9E0>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [17]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'param_max_features', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_max_features,mean_test_score
41,18,5,15,None,0.861514
38,16,7,20,None,0.854662
46,17,10,22,None,0.851546
43,18,11,20,None,0.850332
3,13,10,19,None,0.849870
13,17,7,28,None,0.849031
7,12,8,23,None,0.848106
47,18,5,24,sqrt,0.847994
30,17,7,15,log2,0.847862
2,16,12,19,None,0.847482


In [18]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 18, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 15}
Best CV score: 0.8615139529275295


In [19]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      2652
           1       0.97      0.96      0.97      2529

    accuracy                           0.97      5181
   macro avg       0.97      0.97      0.97      5181
weighted avg       0.97      0.97      0.97      5181



#### Evaluación

In [20]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.79      0.80       663
           1       0.79      0.80      0.79       633

    accuracy                           0.80      1296
   macro avg       0.80      0.80      0.80      1296
weighted avg       0.80      0.80      0.80      1296



#### Guardado

In [21]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/crudos/features_rf.pkl']

### Gradient Boosting

In [22]:
gbc = GradientBoostingClassifier(
    n_estimators=500,
    n_iter_no_change=10,
    random_state=SEED
)

In [26]:
param_grid = {
    'learning_rate': [0.2, 0.3, 0.4, 0.5],
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV 1/5] END .................learning_rate=0.2;, score=0.830 total time=  28.1s
[CV 2/5] END .................learning_rate=0.2;, score=0.798 total time=  22.9s
[CV 3/5] END .................learning_rate=0.2;, score=0.820 total time=  19.3s
[CV 4/5] END .................learning_rate=0.2;, score=0.816 total time=  17.8s
[CV 5/5] END .................learning_rate=0.2;, score=0.810 total time=  17.7s
[CV 1/5] END .................learning_rate=0.3;, score=0.810 total time=  16.5s
[CV 2/5] END .................learning_rate=0.3;, score=0.796 total time=  15.5s
[CV 3/5] END .................learning_rate=0.3;, score=0.813 total time=  16.9s
[CV 4/5] END .................learning_rate=0.3;, score=0.785 total time=   6.5s
[CV 5/5] END .................learning_rate=0.3;, score=0.820 total time=  14.7s
[CV 1/5] END .................learning_rate=0.4;, score=0.805 total time=  10.9s
[CV 2/5] END .................learning_rate=0.4;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.2, 0.3, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [27]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate',  'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_learning_rate,mean_test_score
0,0.2,0.814734
1,0.3,0.804927
2,0.4,0.802748
3,0.5,0.800249


In [28]:
best_model = rnd_search.best_estimator_

In [29]:
best_model.n_estimators_

122

In [30]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.89      0.86      0.87      2652
           1       0.86      0.88      0.87      2529

    accuracy                           0.87      5181
   macro avg       0.87      0.87      0.87      5181
weighted avg       0.87      0.87      0.87      5181



In [31]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       663
           1       0.73      0.74      0.73       633

    accuracy                           0.74      1296
   macro avg       0.74      0.74      0.74      1296
weighted avg       0.74      0.74      0.74      1296



In [32]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_gbc.pkl')

['./modelos_clasicos/modelos/crudos/features_gbc.pkl']

# Datos aumentados

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_melspectrogram_aug.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_melspectrogram_aug.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15023, 47616), (15023,), (3628, 47616), (3628,))

In [5]:
np.random.seed(SEED)

i_samples = np.random.choice(X_train.shape[0], size=5000, replace=False)

### Random Forest

#### Entrenamiento

In [20]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': randint(5, 15),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(10, 20)
}

In [ ]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train[i_samples], y_train[i_samples])

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.637 total time=  23.4s
[CV 2/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.666 total time=  25.2s
[CV 3/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.667 total time=  30.0s
[CV 4/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.633 total time=  24.9s
[CV 5/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.653 total time=  25.3s
[CV 1/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.641 total time=  26.4s
[CV 2/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.670 total time=  26.5s
[CV 3/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.651 total time=  25.2s
[CV 4/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.636 total time=  24.3s
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....001AFB5E3F8C0>, 'min_samples_leaf': <scipy.stats....001AFB5E90050>, 'min_samples_split': <scipy.stats....001AFB5D5FD90>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [27]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
4,12,17,17,0.654755
6,12,15,16,0.653599
8,14,15,27,0.652926
1,12,14,21,0.652096
3,12,14,18,0.652096
7,9,10,26,0.651933
2,14,12,21,0.651768
5,10,14,16,0.651380
0,11,13,27,0.651106
9,13,10,25,0.649908


In [33]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 12, 'min_samples_leaf': 17, 'min_samples_split': 17}
Best CV score: 0.6547548188017183


In [30]:
best_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,12
,min_samples_split,17
,min_samples_leaf,17
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      7712
           1       0.88      0.90      0.89      7311

    accuracy                           0.89     15023
   macro avg       0.89      0.89      0.89     15023
weighted avg       0.89      0.89      0.89     15023



#### Evaluación

In [35]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.71      0.65      1861
           1       0.62      0.50      0.55      1767

    accuracy                           0.61      3628
   macro avg       0.61      0.60      0.60      3628
weighted avg       0.61      0.61      0.60      3628



#### Guardado

In [36]:
os.makedirs(f'./modelos_clasicos/modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/melspec_rf_aug.pkl')

['./modelos_clasicos/modelos/crudos/melspec_rf_aug.pkl']

### Gradient Boost

In [ ]:
gbc = GradientBoostingClassifier(
    n_estimators=500,
    random_state=SEED,
    n_iter_no_change=10,
    max_features='sqrt'
    )

In [ ]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2]
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END ................learning_rate=0.05;, score=0.726 total time= 1.2min
[CV 2/5] END ................learning_rate=0.05;, score=0.755 total time= 2.8min
[CV 3/5] END ................learning_rate=0.05;, score=0.746 total time= 1.7min
[CV 4/5] END ................learning_rate=0.05;, score=0.739 total time=  58.7s
[CV 5/5] END ................learning_rate=0.05;, score=0.770 total time= 2.3min
[CV 1/5] END .................learning_rate=0.1;, score=0.726 total time= 1.2min
[CV 2/5] END .................learning_rate=0.1;, score=0.740 total time= 1.1min
[CV 3/5] END .................learning_rate=0.1;, score=0.731 total time=  51.8s
[CV 4/5] END .................learning_rate=0.1;, score=0.757 total time= 1.1min
[CV 5/5] END .................learning_rate=0.1;, score=0.770 total time= 1.6min
[CV 1/5] END .................learning_rate=0.2;, score=0.749 total time= 1.2min
[CV 2/5] END .................learning_rate=0.2;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.05, 0.1, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [ ]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_learning_rate,mean_test_score
0,0.05,0.747095
1,0.10,0.744499
2,0.20,0.740088


In [ ]:
best_model = rnd_search.best_estimator_

In [ ]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.83      0.83      0.83      2652
           1       0.82      0.82      0.82      2529

    accuracy                           0.83      5181
   macro avg       0.83      0.83      0.83      5181
weighted avg       0.83      0.83      0.83      5181



In [ ]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.68      0.68      0.68       663
           1       0.66      0.66      0.66       633

    accuracy                           0.67      1296
   macro avg       0.67      0.67      0.67      1296
weighted avg       0.67      0.67      0.67      1296



In [ ]:
joblib.dump(gbc, f'./modelos_clasicos/modelos/{dataset}/melspec_gbc.pkl')

['./modelos_clasicos/modelos/crudos/melspec_gbc.pkl']

## Features de Audio

In [8]:
train_data = np.load(f'./dataset/ciclos_procesados/{dataset}/train_features_aug.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ciclos_procesados/{dataset}/test_features_aug.npz')
X_test = test_data['X']
y_test = test_data['y']

In [9]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15023, 46), (15023,), (3628, 46), (3628,))

### Random Forest

#### Entrenamiento

In [10]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(5, 20),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(5, 15),
    'max_features': ['sqrt', 'log2', None]
}

In [11]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=11, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.708 total time=   1.3s
[CV 2/5] END max_depth=11, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.715 total time=   1.2s
[CV 3/5] END max_depth=11, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.700 total time=   1.2s
[CV 4/5] END max_depth=11, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.689 total time=   1.3s
[CV 5/5] END max_depth=11, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.699 total time=   1.2s
[CV 1/5] END max_depth=9, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.708 total time=   7.5s
[CV 2/5] END max_depth=9, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.713 total time=   7.2s
[CV 3/5] END max_depth=9, max_features=None, min_samples_leaf=14, min_samples_split=17;, scor

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0013DFD3FEBA0>, 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....0013DFD474190>, 'min_samples_split': <scipy.stats....0013DFD474050>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [13]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'param_max_features', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_max_features,mean_test_score
26,16,7,15,None,0.727303
37,17,11,15,None,0.723408
8,16,7,26,None,0.722835
40,19,8,27,None,0.722576
15,17,7,28,None,0.722208
27,12,7,15,None,0.721314
35,18,13,29,None,0.720251
48,13,11,20,None,0.719671
7,15,14,26,None,0.719422
43,18,9,20,sqrt,0.719402


In [14]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 16, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 15}
Best CV score: 0.7273025898250951


In [15]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.94      0.95      7712
           1       0.94      0.95      0.94      7311

    accuracy                           0.95     15023
   macro avg       0.95      0.95      0.95     15023
weighted avg       0.95      0.95      0.95     15023



#### Evaluación

In [16]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.77      0.69      1861
           1       0.68      0.50      0.58      1767

    accuracy                           0.64      3628
   macro avg       0.65      0.64      0.63      3628
weighted avg       0.65      0.64      0.64      3628



#### Guardado

In [17]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_rf_aug.pkl')

['./modelos_clasicos/modelos/crudos/features_rf_aug.pkl']

### Gradient Boosting

In [18]:
gbc = GradientBoostingClassifier(
    n_estimators=500,
    n_iter_no_change=10,
    random_state=SEED
)

In [19]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END ................learning_rate=0.05;, score=0.678 total time=  52.1s
[CV 2/5] END ................learning_rate=0.05;, score=0.683 total time= 1.3min
[CV 3/5] END ................learning_rate=0.05;, score=0.690 total time= 1.6min
[CV 4/5] END ................learning_rate=0.05;, score=0.655 total time=  53.0s
[CV 5/5] END ................learning_rate=0.05;, score=0.675 total time= 1.1min
[CV 1/5] END .................learning_rate=0.1;, score=0.685 total time=  45.2s
[CV 2/5] END .................learning_rate=0.1;, score=0.676 total time=  22.8s
[CV 3/5] END .................learning_rate=0.1;, score=0.689 total time=  40.6s
[CV 4/5] END .................learning_rate=0.1;, score=0.661 total time=  32.6s
[CV 5/5] END .................learning_rate=0.1;, score=0.682 total time=  48.6s
[CV 1/5] END .................learning_rate=0.2;, score=0.691 total time=  20.6s
[CV 2/5] END .................learning_rate=0.2;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.05, 0.1, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [20]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate',  'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_learning_rate,mean_test_score
2,0.20,0.679598
1,0.10,0.678413
0,0.05,0.676449


In [21]:
best_model = rnd_search.best_estimator_

In [22]:
best_model.n_estimators_

149

In [23]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.74      0.74      0.74      7712
           1       0.72      0.72      0.72      7311

    accuracy                           0.73     15023
   macro avg       0.73      0.73      0.73     15023
weighted avg       0.73      0.73      0.73     15023



In [24]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.59      0.73      0.65      1861
           1       0.62      0.47      0.54      1767

    accuracy                           0.60      3628
   macro avg       0.61      0.60      0.59      3628
weighted avg       0.61      0.60      0.60      3628



In [25]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_gbc_aug.pkl')

['./modelos_clasicos/modelos/crudos/features_gbc_aug.pkl']